In [1]:
# Given an integer array nums, return an array answer such that 
# answer[i] is equal to the product of all the elements of nums except nums[i].
# The product of any prefix or suffix of nums is guaranteed to fit in a 32bit integer.
# You must write an algorithm that runs in O(n) time and without using the division operation.
# Example walk through 
# Input: nums = [1, 2, 3, 4] Calculation: Index 0 → 2 × 3 × 4 = 24 Index 1 → 1 × 3 × 4 = 12 Index 2 → 1 × 2 × 4 = 8 Index 3 → 1 × 2 × 3 = 6 Output: [24, 12, 8, 6], Test cases
# Input: nums = [1,2,3,4]  Output: [24,12,8,6]
# Input: nums = [-1,1,0,-3,3]  Output: [0,0,9,0,0]

In [120]:
nums = [1, 2, 3, 4]
# nums = [-1,1,0,-3,3]
output_num = []

for i in range(len(nums)):
    nums_copy = nums.copy()
    # print(nums_copy)
    result = 1
    nums_copy.pop(i)
    # print(nums_copy)
    for item in nums_copy:
        result = result*item
    output_num.append(result)

print(output_num)

[24, 12, 8, 6]


In [121]:
nums[3-1]

3

In [24]:
movies = { "movie_id": [1, 2, 3], "title": ["Avengers", "Frozen 2", "Joker"]} 
users = { "user_id": [1, 2, 3, 4], "name": ["Daniel", "Monica", "Maria", "James"]} 
movie_ratings = { 
"movie_id": [1, 1, 1, 1, 2, 2, 2, 3, 3],"user_id":[1, 2, 3, 4, 1, 2, 3, 1, 2], 
"rating":[3, 4, 2, 1, 5, 2, 2, 3, 4], 
"created_at":[ 
"2020-01-12", "2020-02-11", "2020-02-12", "2020-01-01", 
"2020-02-17", "2020-02-01", "2020-03-01", 
"2020-02-22", "2020-02-25" 
] 
} 

import pandas as pd 
import sqlite3 
conn = sqlite3.connect(":memory:") 
df_movies = pd.DataFrame(movies) 
df_users = pd.DataFrame(users) 
df_ratings = pd.DataFrame(movie_ratings) 
df_movies.to_sql("movies", conn, index=False) 
df_users.to_sql("users", conn, index=False) 
df_ratings.to_sql("movieratings", conn, index=False) 

9

In [25]:
df_ratings

,movie_id,user_id,rating,created_at
0,1,1,3,2020-01-12
1,1,2,4,2020-02-11
2,1,3,2,2020-02-12
3,1,4,1,2020-01-01
4,2,1,5,2020-02-17
5,2,2,2,2020-02-01
6,2,3,2,2020-03-01
7,3,1,3,2020-02-22
8,3,2,4,2020-02-25


In [33]:
SELECT mr.title, m.movie_id, AVG(m.rating) from movieratings mr
INNER JOIN movies m ON m.movie_id = mr.movie_id
WHERE created_at BETWEEN(2020-02-01, 2020-02-29) 
GROUP BY movie_id
ORDER BY AVG(m.rating) DESC, mr.title ASC
LIMIT 1

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (311164395.py, line 3)

In [32]:
df_movies

,movie_id,title
0,1,Avengers
1,2,Frozen 2
2,3,Joker


In [38]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np

# Load California housing and create synthetic weekly time series
housing = fetch_california_housing()
np.random.seed(42)

n_weeks, n_regions = 260, 8
date_range = pd.date_range(start='2019-01-07', periods=n_weeks, freq='W')
region_names = ['Bay Area', 'Los Angeles', 'San Diego', 'Sacramento', 
            'Fresno', 'Riverside', 'Orange County', 'Santa Barbara']

# Vectorized generation
regions = np.repeat(np.arange(n_regions), n_weeks)
dates = np.tile(date_range, n_regions)
weeks = np.tile(np.arange(n_weeks), n_regions)

base_prices = np.random.uniform(300000, 800000, n_regions)[regions]
trends = np.random.uniform(50000, 150000, n_regions)[regions] * weeks / n_weeks
seasonality = 20000 * np.sin(2 * np.pi * weeks / 52)
prices = base_prices + trends + seasonality + np.random.normal(0, 15000, len(regions))

base_vol = np.random.uniform(50, 200, n_regions)[regions]
volume = np.maximum(base_vol + 30 * np.sin(2 * np.pi * weeks / 52) + np.random.normal(0, 15, len(regions)), 5).astype(int)

df_ts = pd.DataFrame({
'date': dates, 'region_id': regions, 'region_name': np.array(region_names)[regions],
'median_price': prices, 'sales_volume': volume,
'interest_rate': 3.5 + 0.5 * np.sin(2 * np.pi * weeks / 104) + np.random.normal(0, 0.1, len(regions)),
'inventory': np.random.poisson(500, len(regions)) + 100 * np.sin(2 * np.pi * weeks / 52),
'days_on_market': np.random.exponential(30, len(regions)) + 10
}).sort_values(['region_id', 'date']).reset_index(drop=True)

print(f"Shape: {df_ts.shape} | Date range: {df_ts['date'].min().date()} to {df_ts['date'].max().date()}")
print(f"Regions: {df_ts['region_name'].unique().tolist()}")

Shape: (2080, 8) | Date range: 2019-01-13 to 2023-12-31
Regions: ['Bay Area', 'Los Angeles', 'San Diego', 'Sacramento', 'Fresno', 'Riverside', 'Orange County', 'Santa Barbara']


In [193]:
df_ts.shape

(2080, 8)

In [171]:
def EWMA_formula(alpha, current_median_price, prev_EWMA):
    EWMA = alpha * current_median_price + (1 - alpha) * prev_EWMA
    return EWMA

def calc_EWMA(df, region, alpha):
    df_temp = df[df["region_name"]==region].sort_values(by = "date").reset_index(drop=True)
    EWMA_lst = []
    col_name = "EWMA_median_price_alpha_"+str(alpha)
    df_temp[col_name] = None
    for idx, row in df_temp.iterrows(): 
        if idx == 0:
            EWMA = row["median_price"]
            EWMA_lst.append(EWMA)
            df_temp.at[idx, col_name] = EWMA
        else:
            EWMA = EWMA_formula(alpha, row["median_price"], EWMA_lst[idx-1])
            EWMA_lst.append(EWMA)
            df_temp.at[idx, col_name] = EWMA
    return df_temp

In [192]:
alpha_lst = [0.90, 0.95, 0.99]
region_name = df_ts["region_name"].unique()
df_output = []

for region in region_name:
    df_region_output = pd.DataFrame()
    for alpha in alpha_lst:
        output = calc_EWMA(df_ts,region, alpha)
        if df_region_output.empty:
            df_region_output = output.copy()
        else:
            col_name = "EWMA_median_price_alpha_"+str(alpha)
            df_region_output[col_name] = output[col_name]
    df_output.append(df_region_output)

df_output = pd.concat(df_output).reset_index(drop=True)
df_output

,date,region_id,region_name,median_price,sales_volume,interest_rate,inventory,days_on_market,EWMA_median_price_alpha_0.9,EWMA_median_price_alpha_0.95,EWMA_median_price_alpha_0.99
0,2019-01-13,0,Bay Area,490899.493497,87,3.527835,508.000000,64.222780,490899.493497,490899.493497,490899.493497
1,2019-01-20,0,Bay Area,461405.095133,71,3.479268,554.053668,52.619862,464354.534969,462879.815051,461700.039116
2,2019-01-27,0,Bay Area,467029.616769,75,3.475632,574.931566,21.625301,466762.108589,466822.126683,466976.320993
3,2019-02-03,0,Bay Area,487198.361547,109,3.531637,542.460489,13.261624,485154.736251,486179.549804,486996.141142
4,2019-02-10,0,Bay Area,483066.079155,77,3.571782,579.472317,12.235485,483274.944864,483221.752687,483105.379774
5,2019-02-17,0,Bay Area,505462.593216,123,3.684699,567.806475,12.970619,503243.828381,504350.55119,505239.021082
6,2019-02-24,0,Bay Area,489453.186098,88,3.912210,539.312266,36.230675,490832.250326,490198.054353,489611.044448
7,2019-03-03,0,Bay Area,484020.259283,113,3.633602,604.851075,11.591335,484701.458388,484329.149037,484076.167135
8,2019-03-10,0,Bay Area,529102.514465,101,3.690699,553.298387,22.326100,524662.408858,526863.846194,528652.250992
9,2019-03-17,0,Bay Area,505404.087393,117,3.752357,594.545603,51.975273,507329.91954,506477.075333,505636.569029


In [37]:
# What is EWMA?
# The Exponential Weighted Moving Average (EWMA) is a statistical technique used to find trends in time-series data. Unlike a simple moving average, which treats all data points equally, EWMA gives more weight to recent observations. This helps capture more current trends, making it useful in various fields, including finance, stock prediction, and especially in deep learning optimizers like Adam.
 
# How Does EWMA Work?
# To calculate EWMA, we assign a decay rate or smoothing factor (usually denoted by α), a value between 0 and 1. The smoothing factor controls how much weight is given to recent values versus past values.
# The formula for EWMA at any time t is:
# x(t)  is the current data point.
 
# α is the smoothing factor..
 
# EWMA(t−1)  is the previous EWMA value.
 
# The higher the value of α, the more weight given to recent data. For example, if α = 0.8, 80% of the EWMA is based on the latest data point, making the EWMA highly responsive to new information.
 
# Example of EWMA Calculation
# Let’s say we have a series of daily stock prices over the last 5 days:
# Day 1: 100
# Day 2: 105
# Day 3: 103
# Day 4: 108
# Day 5: 107
# Assume we start with an initial average (say, the first data point, 100), and let α = 0.5 
 
# Using the formula, here’s how the EWMA evolves each day:
# Day 1: EWMA = 100 (initial value)
# Day 2: EWMA = 0.5 * 105 + (1–0.5) * 100 = 102.5
# Day 3: EWMA = 0.5 * 103 + (1–0.5) * 102.5 = 102.75
# Day 4: EWMA = 0.5 * 108 + (1–0.5) * 102.75 = 105.375
# Day 5: EWMA = 0.5 * 107 + (1–0.5) * 105.375 = 106.1875

In [ ]:
# Q1: For each region, compute EWMA of 'median_price' with multiple alpha values (0.90, 0.95, 0.99). Create separate columns for each alpha. Ensure proper handling of the first observation in each region's time series.